# Day 4, Part 3: Embeddings — A Universal Tool for Any Kind of Data

**MIDAS Summer Academy**

In this notebook, you will:
1. Learn what embeddings are and why they're useful across all scientific domains
2. Embed and cluster **text** (research abstracts) to discover hidden structure
3. Embed and cluster **images** using a vision model
4. Embed and cluster **audio** using a sound model
5. See how one model (CLIP) can connect text and images in the same space

### The Big Idea

An **embedding model** converts any input — a sentence, a photo, a sound clip — into a list of numbers (a **vector**). Inputs that are *similar* end up with vectors that are *close together*.

This means you can use the same tools (similarity, clustering, visualization) on **any kind of data**, as long as you have a model that embeds it.

**No API keys required for this notebook** — all models run locally in Colab.

## Setup

In [ ]:
!pip install -q sentence-transformers transformers umap-learn datasets torch torchvision torchaudio scikit-learn matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("Setup complete!")

---
## 1. What Are Embeddings?

### Analogy: GPS Coordinates for Meaning

GPS gives every place on Earth a pair of numbers (latitude, longitude). Places that are *near each other* in the real world have *similar coordinates*.

Embeddings do the same thing for data. Instead of 2 numbers, an embedding might have 384 or 768 numbers — but the principle is identical:

- A photo of a **cat** and a photo of a **kitten** → nearby vectors
- A photo of a **cat** and a photo of a **truck** → distant vectors
- A sentence about **protein folding** and one about **molecular structure** → nearby vectors
- A sentence about **protein folding** and one about **tax policy** → distant vectors

### Why Scientists Care

| Use Case | What Embeddings Let You Do |
|---|---|
| **Cluster abstracts** | Discover research themes automatically |
| **Find similar images** | Group microscopy images, satellite photos, or specimens |
| **Classify sounds** | Sort field recordings by species or event type |
| **Detect outliers** | Find anomalous data points that don't fit any cluster |
| **Search by meaning** | Find relevant items even with different terminology |

---
## 2. Text Embeddings and Clustering

Let's start with text — the most common use case. We have 20 research abstracts from diverse fields. Can we use embeddings to automatically discover which abstracts are about similar topics?

### 2.1 The Data

In [ ]:
abstracts = [
    # --- Climate & Earth Science ---
    "Analysis of ice cores from Greenland reveals abrupt climate shifts occurring within decades during the last glacial period, challenging gradual-change models.",
    "Satellite observations show that Arctic sea ice extent reached a new record low in September 2024, with ice-free summers now projected by the 2050s.",
    "A coupled ocean-atmosphere model predicts intensification of El Niño events under doubled CO2 scenarios, with cascading effects on global precipitation.",
    "Analysis of 50 years of tide gauge records shows accelerating sea level rise along the U.S. Atlantic coast, exceeding global mean rates by 30%.",

    # --- Genomics & Molecular Biology ---
    "Single-cell RNA sequencing of human liver tissue reveals 12 previously unknown hepatocyte subtypes with distinct metabolic gene expression signatures.",
    "CRISPR-Cas9 base editing corrects the sickle cell mutation in patient-derived stem cells with 89% efficiency and minimal off-target effects.",
    "Long-read nanopore sequencing of the axolotl genome identifies novel gene families involved in limb regeneration not found in other vertebrates.",
    "Spatial transcriptomics mapping of mouse brain reveals cell-type-specific gene expression gradients along the hippocampal axis.",

    # --- Ecology & Conservation ---
    "Camera trap surveys across 50 protected areas show that large carnivore populations have declined 40% over two decades despite formal protection.",
    "Environmental DNA sampling of river water detects 23 fish species, including 3 thought locally extinct, outperforming traditional electrofishing surveys.",
    "Pollinator network analysis reveals that loss of a single keystone bumblebee species cascades to affect reproduction of 35% of local wildflower species.",
    "Acoustic monitoring of tropical forests shows that soundscape diversity correlates strongly with invertebrate biomass and overall ecosystem health.",

    # --- Materials Science & Chemistry ---
    "A novel metal-organic framework achieves record CO2 capture capacity of 8.2 mmol/g under humid flue gas conditions with full regeneration at 80°C.",
    "Perovskite-silicon tandem solar cells reach 33.7% power conversion efficiency, surpassing the theoretical single-junction silicon limit.",
    "Machine-learning-guided high-throughput screening identifies three new high-entropy alloys with exceptional strength-to-weight ratios for aerospace applications.",
    "Biodegradable polymer scaffolds seeded with mesenchymal stem cells promote complete bone regeneration in critical-sized defects within 12 weeks.",

    # --- Public Health & Epidemiology ---
    "A cluster-randomized trial in 200 villages demonstrates that community health worker programs reduce under-5 mortality by 28% in rural sub-Saharan Africa.",
    "Wastewater surveillance detects emerging SARS-CoV-2 variants 10-14 days before they appear in clinical testing data across 30 U.S. cities.",
    "Longitudinal cohort study of 50,000 adults finds that ultra-processed food consumption above 4 servings per day is associated with 31% higher all-cause mortality.",
    "Mathematical modeling of antimicrobial resistance spread predicts that without intervention, drug-resistant infections will cause 10 million deaths annually by 2050.",
]

# Human-assigned labels (we'll see if clustering recovers them)
true_labels = (
    ["Climate"] * 4
    + ["Genomics"] * 4
    + ["Ecology"] * 4
    + ["Materials"] * 4
    + ["Public Health"] * 4
)

print(f"Loaded {len(abstracts)} abstracts from {len(set(true_labels))} fields.")

### 2.2 Compute Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

# Load a small, fast embedding model (~80 MB download)
text_model = SentenceTransformer("all-MiniLM-L6-v2")

# Embed all 20 abstracts
text_embeddings = text_model.encode(abstracts)

print(f"Each abstract is now a vector of {text_embeddings.shape[1]} numbers.")
print(f"Matrix shape: {text_embeddings.shape}  (20 abstracts × 384 dimensions)")

### 2.3 Check: Do Similar Abstracts Have Similar Embeddings?

We measure similarity with **cosine similarity** — the cosine of the angle between two vectors. It ranges from -1 (opposite) to 1 (identical).

In [ ]:
# Compare two climate abstracts (should be similar)
sim_same = cosine_similarity(text_embeddings[0], text_embeddings[1])

# Compare a climate abstract to a genomics abstract (should be different)
sim_diff = cosine_similarity(text_embeddings[0], text_embeddings[4])

print(f"Two climate abstracts:               {sim_same:.3f}")
print(f"Climate vs. genomics abstract:        {sim_diff:.3f}")

### 2.4 Visualize with UMAP

Our embeddings have 384 dimensions — we can't plot that. **UMAP** (Uniform Manifold Approximation and Projection) is a dimensionality reduction technique that squashes high-dimensional data down to 2D while preserving the neighborhood structure. Similar points stay near each other.

In [ ]:
import umap

# Reduce 384 dimensions → 2 dimensions for plotting
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=5, min_dist=0.3)
text_2d = reducer.fit_transform(text_embeddings)

# Plot, colored by true labels
fig, ax = plt.subplots(figsize=(8, 6))

colors = {"Climate": "#e74c3c", "Genomics": "#3498db", "Ecology": "#2ecc71",
          "Materials": "#f39c12", "Public Health": "#9b59b6"}

for label in colors:
    mask = [l == label for l in true_labels]
    ax.scatter(text_2d[mask, 0], text_2d[mask, 1],
              c=colors[label], label=label, s=80, alpha=0.8, edgecolors="white")

ax.legend(title="Field")
ax.set_title("Research Abstracts in Embedding Space (UMAP projection)")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
plt.tight_layout()
plt.show()

Abstracts from the same field should cluster together — the embedding model has never seen our labels, yet it puts related papers nearby based purely on their content.

### 2.5 Clustering with K-Means

Now let's see if an algorithm can recover these groups **without** knowing our labels. K-Means assigns each point to one of *k* clusters by finding natural groupings in the embedding space.

In [ ]:
# Cluster the embeddings into 5 groups
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(text_embeddings)

# Show which abstracts ended up in which cluster
for cluster_id in range(5):
    print(f"\n--- Cluster {cluster_id} ---")
    for i, (abstract, true_label) in enumerate(zip(abstracts, true_labels)):
        if cluster_labels[i] == cluster_id:
            print(f"  [{true_label:13s}] {abstract[:80]}...")

In [ ]:
# Visualize clusters vs true labels side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: true labels
for label in colors:
    mask = [l == label for l in true_labels]
    ax1.scatter(text_2d[mask, 0], text_2d[mask, 1],
               c=colors[label], label=label, s=80, alpha=0.8, edgecolors="white")
ax1.legend(title="True Field")
ax1.set_title("Colored by True Labels")

# Right: K-means clusters
scatter = ax2.scatter(text_2d[:, 0], text_2d[:, 1],
                      c=cluster_labels, cmap="Set1", s=80, alpha=0.8, edgecolors="white")
ax2.legend(*scatter.legend_elements(), title="Cluster")
ax2.set_title("Colored by K-Means Clusters")

plt.tight_layout()
plt.show()

**How well did unsupervised clustering recover the true fields?** The clusters won't have the same color assignments, but you should see that most abstracts from the same field land in the same cluster.

### Exercise 1

Add 4 abstracts of your own — either from your research field or made-up summaries. Re-run the embedding, UMAP, and clustering steps. Do your new abstracts cluster together? Do they land near any of the existing groups?

In [ ]:
# YOUR CODE HERE
# Hint: add your abstracts to the list, update true_labels, and re-run the cells above.

# my_abstracts = [
#     "...",
#     "...",
#     "...",
#     "...",
# ]
# all_abstracts = abstracts + my_abstracts
# all_labels = true_labels + ["My Field"] * 4
# all_embeddings = text_model.encode(all_abstracts)

---
## 3. Image Embeddings and Clustering

The same idea works for images. We'll use **CLIP** (Contrastive Language-Image Pre-training), a model from OpenAI that was trained to understand both text and images. It can produce an embedding for any image.

### 3.1 Load Images

We'll use a subset of the CIFAR-10 dataset — 10,000 tiny images in 10 categories (airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck). We'll sample a few from each category to keep things fast.

In [ ]:
from datasets import load_dataset
from PIL import Image

# Load CIFAR-10 test set from HuggingFace
cifar = load_dataset("cifar10", split="test")

# Class names in CIFAR-10
class_names = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

# Pick 5 categories and sample 15 images from each (75 total)
selected_classes = ["bird", "cat", "dog", "ship", "truck"]
selected_class_ids = [class_names.index(c) for c in selected_classes]

images = []
image_labels = []

for class_id, class_name in zip(selected_class_ids, selected_classes):
    class_images = [ex for ex in cifar if ex["label"] == class_id]
    for ex in class_images[:15]:
        images.append(ex["img"])
        image_labels.append(class_name)

print(f"Selected {len(images)} images from {len(selected_classes)} categories.")

# Show a few examples
fig, axes = plt.subplots(1, 5, figsize=(12, 2.5))
for ax, cls in zip(axes, selected_classes):
    idx = image_labels.index(cls)
    ax.imshow(images[idx])
    ax.set_title(cls)
    ax.axis("off")
plt.suptitle("One example from each category")
plt.tight_layout()
plt.show()

### 3.2 Embed Images with CLIP

In [ ]:
from transformers import CLIPModel, CLIPProcessor

# Load CLIP (downloads ~600 MB on first run)
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

print("CLIP model loaded!")

In [ ]:
import torch

# Embed all images in batches
image_embeddings = []
batch_size = 25

for i in range(0, len(images), batch_size):
    batch = images[i:i + batch_size]
    inputs = clip_processor(images=batch, return_tensors="pt", padding=True)
    with torch.no_grad():
        features = clip_model.get_image_features(**inputs)
    image_embeddings.append(features.numpy())

image_embeddings = np.vstack(image_embeddings)

# Normalize embeddings (CLIP embeddings work best normalized)
image_embeddings = image_embeddings / np.linalg.norm(image_embeddings, axis=1, keepdims=True)

print(f"Image embeddings shape: {image_embeddings.shape}  ({len(images)} images × {image_embeddings.shape[1]} dimensions)")

### 3.3 Cluster and Visualize

In [ ]:
# Reduce to 2D with UMAP
reducer_img = umap.UMAP(n_components=2, random_state=42, n_neighbors=10, min_dist=0.3)
img_2d = reducer_img.fit_transform(image_embeddings)

# K-Means clustering
img_kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
img_cluster_labels = img_kmeans.fit_predict(image_embeddings)

# Plot: true labels vs clusters
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

img_colors = {"bird": "#e74c3c", "cat": "#3498db", "dog": "#2ecc71",
              "ship": "#f39c12", "truck": "#9b59b6"}

for label in img_colors:
    mask = [l == label for l in image_labels]
    ax1.scatter(img_2d[mask, 0], img_2d[mask, 1],
               c=img_colors[label], label=label, s=50, alpha=0.7, edgecolors="white")
ax1.legend(title="True Category")
ax1.set_title("Images Colored by True Category")

scatter = ax2.scatter(img_2d[:, 0], img_2d[:, 1],
                      c=img_cluster_labels, cmap="Set1", s=50, alpha=0.7, edgecolors="white")
ax2.legend(*scatter.legend_elements(), title="Cluster")
ax2.set_title("Images Colored by K-Means Clusters")

plt.tight_layout()
plt.show()

Notice how the animals (bird, cat, dog) tend to cluster closer to each other than to vehicles (ship, truck). The model understands visual similarity at a semantic level — it's not just matching pixel colors.

### Exercise 2

Look at the image clustering results.
1. Which categories are easiest for the model to separate? Which get confused?
2. Why might that be? (Think about what these objects look like.)
3. Try changing `selected_classes` to a different set of 5 categories and re-run. Does the clustering get easier or harder with different categories?

In [ ]:
# YOUR CODE HERE
# Available classes: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck


---
## 4. Audio Embeddings and Clustering

Embeddings aren't limited to text and images — they work on **sound** too. We'll use a pre-trained audio classification model to embed environmental sounds, then cluster them.

### 4.1 Load Audio Data

We'll use the **ESC-50** dataset — a collection of 50 categories of environmental sounds (rain, dog bark, chainsaw, clock tick, etc.). We'll work with a small subset.

In [ ]:
from datasets import load_dataset, Audio

# Load a small environmental sound dataset
esc50 = load_dataset("ashraq/esc50", split="train")

# Pick 4 sound categories with clear differences
selected_sounds = ["rain", "dog", "chainsaw", "clock_tick"]

audio_samples = []
audio_labels = []

for sound_name in selected_sounds:
    matching = [ex for ex in esc50 if ex["category"] == sound_name]
    for ex in matching[:8]:  # 8 clips per category
        audio_samples.append(ex["audio"])
        audio_labels.append(sound_name)

print(f"Loaded {len(audio_samples)} audio clips from {len(selected_sounds)} categories.")
print(f"Categories: {selected_sounds}")

In [ ]:
# Listen to one example from each category
from IPython.display import Audio as IPAudio, display

for sound_name in selected_sounds:
    idx = audio_labels.index(sound_name)
    sample = audio_samples[idx]
    print(f"\n{sound_name}:")
    display(IPAudio(sample["array"], rate=sample["sampling_rate"]))

### 4.2 Embed Audio

We'll use a pre-trained audio model to convert each sound clip into an embedding vector. The model was trained to recognize many types of sounds, so it has learned a rich internal representation.

In [ ]:
from transformers import ASTModel, ASTFeatureExtractor
import torch

# Load the Audio Spectrogram Transformer (AST)
ast_model = ASTModel.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
ast_extractor = ASTFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")

print("Audio model loaded!")

In [ ]:
import librosa

# Embed each audio clip
audio_embeddings = []

for i, sample in enumerate(audio_samples):
    waveform = sample["array"]
    sr = sample["sampling_rate"]

    # Resample to 16kHz if needed (the model expects 16kHz)
    if sr != 16000:
        waveform = librosa.resample(waveform, orig_sr=sr, target_sr=16000)

    # Process and embed
    inputs = ast_extractor(waveform, sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        outputs = ast_model(**inputs)

    # Use the [CLS] token embedding as the clip's representation
    embedding = outputs.last_hidden_state[:, 0, :].squeeze().numpy()
    audio_embeddings.append(embedding)

audio_embeddings = np.array(audio_embeddings)
print(f"Audio embeddings shape: {audio_embeddings.shape}")

### 4.3 Cluster and Visualize

In [ ]:
# UMAP
reducer_audio = umap.UMAP(n_components=2, random_state=42, n_neighbors=5, min_dist=0.3)
audio_2d = reducer_audio.fit_transform(audio_embeddings)

# K-Means
audio_kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
audio_cluster_labels = audio_kmeans.fit_predict(audio_embeddings)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

audio_colors = {"rain": "#3498db", "dog": "#e74c3c", "chainsaw": "#f39c12", "clock_tick": "#2ecc71"}

for label in audio_colors:
    mask = [l == label for l in audio_labels]
    ax1.scatter(audio_2d[mask, 0], audio_2d[mask, 1],
               c=audio_colors[label], label=label, s=80, alpha=0.8, edgecolors="white")
ax1.legend(title="Sound Type")
ax1.set_title("Sounds Colored by True Category")

scatter = ax2.scatter(audio_2d[:, 0], audio_2d[:, 1],
                      c=audio_cluster_labels, cmap="Set1", s=80, alpha=0.8, edgecolors="white")
ax2.legend(*scatter.legend_elements(), title="Cluster")
ax2.set_title("Sounds Colored by K-Means Clusters")

plt.tight_layout()
plt.show()

The same workflow — embed, reduce, cluster — applied to a completely different data type. The model learned to represent sounds in a way that groups similar sounds together, just as the text model grouped similar abstracts.

---
## 5. Cross-Modal Embeddings: Connecting Text and Images

Here's where things get really interesting. **CLIP** was trained on text-image pairs, so it embeds text and images into the **same vector space**. This means you can:
- Compare a text description to an image
- Search for images using a text query

Let's use this to search our CIFAR images with natural language.

In [ ]:
def clip_text_embed(texts):
    """Embed text using CLIP's text encoder."""
    inputs = clip_processor(text=texts, return_tensors="pt", padding=True)
    with torch.no_grad():
        features = clip_model.get_text_features(**inputs)
    emb = features.numpy()
    return emb / np.linalg.norm(emb, axis=1, keepdims=True)


def search_images(query, top_k=5):
    """Search images using a text query via CLIP embeddings."""
    query_emb = clip_text_embed([query])[0]

    # Cosine similarity to all image embeddings
    similarities = image_embeddings @ query_emb

    top_indices = np.argsort(similarities)[::-1][:top_k]
    return top_indices, similarities[top_indices]

In [ ]:
# Search for images with a text description
query = "a small animal with fur"

indices, scores = search_images(query, top_k=8)

fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
fig.suptitle(f'Search results for: "{query}"', fontsize=13)

for ax, idx, score in zip(axes, indices, scores):
    ax.imshow(images[idx])
    ax.set_title(f"{image_labels[idx]}\n{score:.2f}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Try another query
query = "a vehicle on water"

indices, scores = search_images(query, top_k=8)

fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
fig.suptitle(f'Search results for: "{query}"', fontsize=13)

for ax, idx, score in zip(axes, indices, scores):
    ax.imshow(images[idx])
    ax.set_title(f"{image_labels[idx]}\n{score:.2f}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

### Exercise 3

Try 3-4 text queries of your own. Some ideas:
- Try a query that doesn't exactly match any category name (e.g., "pet" or "transportation")
- Try a query about color or shape (e.g., "something red" or "something with wings")
- What are the limits of cross-modal search on these tiny 32×32 images?

In [ ]:
# YOUR CODE HERE

# query = "your query here"
# indices, scores = search_images(query, top_k=8)
#
# fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
# fig.suptitle(f'Search results for: "{query}"', fontsize=13)
# for ax, idx, score in zip(axes, indices, scores):
#     ax.imshow(images[idx])
#     ax.set_title(f"{image_labels[idx]}\n{score:.2f}", fontsize=9)
#     ax.axis("off")
# plt.tight_layout()
# plt.show()

---
## 6. The Universality of Embeddings

We've now applied the **exact same workflow** — embed → visualize → cluster → search — to three completely different data types:

| Data Type | Model | Embedding Dim | What It Captures |
|---|---|---|---|
| **Text** | all-MiniLM-L6-v2 | 384 | Meaning and topic |
| **Images** | CLIP ViT-B/32 | 512 | Visual content and semantics |
| **Audio** | AST (AudioSet) | 768 | Sound characteristics |

This pattern generalizes further. Researchers have built embedding models for:

| Domain | What Gets Embedded | Example Use |
|---|---|---|
| Proteins | Amino acid sequences | Predict function from sequence (ESM-2) |
| Molecules | Chemical structures | Find drugs with similar properties |
| Geospatial | Satellite images | Cluster land use types |
| Medical | Pathology slides | Group similar tissue patterns |
| Music | Audio signals | Recommend similar songs |
| Graphs | Network structures | Detect communities in social or biological networks |

**The core insight:** Once your data is in embedding space, you have access to a universal toolkit — similarity search, clustering, visualization, outlier detection, classification — regardless of the original data type.

> **Discussion question:** What kind of data in your research could benefit from embedding and clustering? What would you hope to discover?

---
## Summary

| What We Covered | Key Takeaway |
|---|---|
| Embeddings | Vectors that capture meaning; similar inputs → nearby vectors |
| Text clustering | Automatically discover research themes from abstracts |
| Image clustering | Group images by visual content, not just pixels |
| Audio clustering | Group sounds by acoustic similarity |
| Cross-modal search | CLIP connects text and images in one space |
| Universality | Same embed→cluster→visualize workflow for any data type |

### The Big Picture: What We Learned Today

| Part | Topic | Key Skill |
|---|---|---|
| 1 | Foundation models + HuggingFace | Use pre-trained models for common tasks |
| 2 | LLMs for data processing | Extract structured data from unstructured text at scale |
| 3 | Embeddings + clustering | Represent *any* data as vectors; discover hidden structure |

These three skills form the practical foundation for using AI in scientific research today.